# MaveDB metadata curation workflow

This notebook assembles the sealed LLM metadata export with optional human curation, processes MaveDB datasets, combines the per-dataset parquet files, then standardizes and validates the combined metadata.

Run cells in order. Update the paths in **Run inputs** first. Each run writes to a new directory; choose a new `RUN_NAME` when rerunning. BigQuery publishing is a separate, disabled-by-default step at the end.

For a non-interactive run, see `curation_tools/study_curation/mavedb/README.md`.


## Setup

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display


REPO_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data_exploration" / "curation_tools").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from within the PerturbationCatalogue repository")

sys.path.insert(0, str(REPO_ROOT / "data_exploration"))

from curation_tools.study_curation.mavedb import (
    combine_dataset_parquets,
    create_run_directory,
    filter_curated_metadata,
    finalize_outputs,
    load_and_merge_metadata,
    process_mavedb_datasets,
    publish_to_bigquery,
    standardize_metadata,
)

from curation_tools.study_curation.paths import (
    CURATION_RUNS_DIR,
    DEFAULT_MAPPING_PATH,
    DEFAULT_OVERRIDE_PATH,
    MAVEDB_DIR,
    MAVEDB_DUMP_DIR,
)


## Run inputs

In [ ]:
MAVEDB_ROOT = MAVEDB_DIR
RUNS_DIR = CURATION_RUNS_DIR
RUN_NAME = "my-new-run"  # choose the sealed LLM run to process
PROCESSING_NAME = "default"  # choose a fresh name for each postprocessing run
BASE_DIR = RUNS_DIR / RUN_NAME
OUTPUT_DIR = BASE_DIR / "postprocessing" / PROCESSING_NAME

# Select the CSV exported from a sealed LLM curation run.
LLM_METADATA_PATH = BASE_DIR / "exports" / "curation_final_metadata.csv"

# Set to None when running without the human curation spreadsheet.
MANUAL_METADATA_PATH = MAVEDB_ROOT / "mavedb_studies.xlsx"

# Controlled term mappings used during metadata standardization.
MAPPING_PATH = DEFAULT_MAPPING_PATH
OVERRIDE_PATH = DEFAULT_OVERRIDE_PATH
MAVEDB_CSV_DIR = MAVEDB_DUMP_DIR
BIGQUERY_PROJECT = "prj-ext-dev-pertcat-437314"


## 1. Assemble metadata sources

Non-empty human values take precedence. Conflicting values are returned for review; LLM-only dataset rows and columns are retained.

In [ ]:
merged_metadata, source_conflicts = load_and_merge_metadata(
    llm_metadata_path=LLM_METADATA_PATH,
    manual_metadata_path=MANUAL_METADATA_PATH,
)

# Filter the merged metadata to include only curated datasets (by human or LLM; i.e., those having a "is_curated" flag) for further processing
datasets_for_processing = filter_curated_metadata(merged_metadata)
print(f"Merged datasets: {merged_metadata['dataset_id'].nunique():,}")
print(f"Curated datasets selected for processing: {datasets_for_processing['dataset_id'].nunique():,}")
print(f"Values present in both human and LLM metadata (just for inspection, as human values take precedence): {len(source_conflicts):,}")
display(source_conflicts.head(50))
print("Conflicting fields (by count):")
display(source_conflicts['field'].value_counts())


## 2. Save run inputs and process datasets

The output directory must be new or empty. Per-dataset progress is shown here; detailed processing messages go to `logs/per_dataset_processing.log`.

In [ ]:
RUN_DIR = create_run_directory(OUTPUT_DIR)
for subdirectory in ("sources", "normalized", "qc"):
    (RUN_DIR / subdirectory).mkdir(parents=True, exist_ok=True)

merged_metadata.to_csv(RUN_DIR / "sources" / "merged_metadata.csv", index=False)
datasets_for_processing.to_csv(
    RUN_DIR / "sources" / "metadata_for_processing.csv", index=False
)
source_conflicts.to_csv(RUN_DIR / "qc" / "source_conflicts.csv", index=False)

processing_log = process_mavedb_datasets(
    metadata=datasets_for_processing,
    mavedb_csv_dir=MAVEDB_CSV_DIR,
    output_dir=RUN_DIR,
)
print(f"Detailed log: {processing_log}")

## 3. Combine per-dataset parquet files

In [ ]:
# Comment out when troubleshooting 

# RUN_DIR = OUTPUT_DIR

In [ ]:
combined_metadata_path, combined_data_path = combine_dataset_parquets(RUN_DIR)

combined_metadata = pd.read_parquet(combined_metadata_path)
combined_data = pd.read_parquet(combined_data_path)
print(f"Combined metadata rows: {len(combined_metadata):,}")
print(f"Combined score rows: {len(combined_data):,}")


## 4. Standardize and review QC tables

Term mappings and study-specific corrections run on the combined metadata. Unmapped values remain unchanged and are saved for review. A dataset-level CSV without `sample_id` and `perturbation_name` is also saved as a reference when editing the mapping and override tables.

In [ ]:
standardized_metadata, unmapped_terms, override_audit = standardize_metadata(
    combined_metadata,
    mapping_path=MAPPING_PATH,
    override_path=OVERRIDE_PATH,
    inplace=True,
)
standardized_metadata_path = (
    RUN_DIR / "normalized" / "mavedb_all_curated_metadata_standardized.parquet"
)
standardized_metadata.to_parquet(standardized_metadata_path, index=False)
dataset_level_metadata = (
    standardized_metadata.drop(
        columns=["sample_id", "perturbation_name"], errors="ignore"
    ).drop_duplicates()
)
dataset_level_metadata_path = (
    RUN_DIR
    / "normalized"
    / "mavedb_all_curated_metadata_standardized_per_dataset.csv"
)
dataset_level_metadata.to_csv(dataset_level_metadata_path, index=False)
print(
    f"Dataset-level metadata CSV ({len(dataset_level_metadata):,} rows): "
    f"{dataset_level_metadata_path}"
)
unmapped_terms.to_csv(RUN_DIR / "qc" / "unmapped_terms.csv", index=False)
override_audit.to_csv(RUN_DIR / "qc" / "override_audit.csv", index=False)

print(f"Unmapped term rows: {len(unmapped_terms):,}")
display(unmapped_terms.head(100))
display(override_audit.loc[override_audit["rows_affected"].gt(0)])


## 5. Validate metadata and align score data

Validation failures display a compact set of failing records and leave the pre-validation parquet available for review.

In [ ]:
# Comment out if troubleshooting 

# def replace_values(df, column, old_value, new_value):
#     mask = df[column] == old_value
#     if mask.sum() == 0:
#         print(f"No occurrences of '{old_value}' found in column '{column}'.")
#         return df
#     else:
#         df.loc[mask, column] = new_value
#         print(f"Replaced {mask.sum()} occurrences of '{old_value}' with '{new_value}' in column '{column}'.")
#         return df
    
# def map_values(df, source_column, target_column, source_value, target_value):
#     mask = df[source_column] == source_value
#     if mask.sum() == 0:
#         print(f"No occurrences of '{source_value}' found in column '{source_column}'.")
#         return df
#     else:
#         df.loc[mask, target_column] = target_value
#         print(f"Mapped {mask.sum()} occurrences of '{source_value}' from column '{source_column}' to '{target_value}' in column '{target_column}'.")
#         return df


# standardized_metadata = replace_values(standardized_metadata, 'species', 'Mus Musculus', 'Mus musculus')
# standardized_metadata = replace_values(standardized_metadata, 'species', "Streptococcus sp. 'group G'", "Streptococcus sp. group G")

# standardized_metadata = map_values(standardized_metadata, 'dataset_id', 'study_uri', 'urn:mavedb:00000010-a-1', 'https://doi.org/10.1261/rna.040709.113')
# standardized_metadata = map_values(standardized_metadata, 'dataset_id', 'study_year', 'urn:mavedb:00000010-a-1', 2013)


In [ ]:
# Comment out when troubleshooting 

# import importlib
# import curation_tools.perturbseq_anndata_schema
# importlib.reload(curation_tools.perturbseq_anndata_schema)
# from curation_tools.perturbseq_anndata_schema import ObsSchema
# from curation_tools.study_curation.mavedb.standardization import (
#     validate_with_unique_values,
# )

# validate_with_unique_values(
#     ObsSchema,
#     standardized_metadata,
#     n_failure_cases=None,
#     lazy=True
# )

In [ ]:
combined_data_path = (RUN_DIR / "final" / "mavedb_all_curated_data_postfilter.parquet")
standardized_metadata_path = (RUN_DIR / "final" / "mavedb_all_curated_metadata_postvalidation.parquet")

artifacts = finalize_outputs(
    standardized_metadata_path=standardized_metadata_path,
    combined_data_path=combined_data_path,
    save_joint_artifact=True,
    output_dir=RUN_DIR,
)

print(f"Validated metadata: {artifacts.metadata_path}")
print(f"Filtered score data: {artifacts.data_path}")
final_metadata = pd.read_parquet(artifacts.metadata_path)
print(f"Validated metadata rows: {len(final_metadata):,}")
display(final_metadata.head())


## 6. Optional BigQuery publish

Review both local parquet outputs before enabling this cell. It checks both destination schemas before writing, then merges rows into the existing MaveDB tables.


#### Filter by species == Homo sapiens

In [ ]:
import polars as pl

meta_hs = (
    pl.scan_parquet((RUN_DIR / "final" / "mavedb_all_curated_metadata_postvalidation.parquet"))
    .filter(pl.col("species").eq("Homo sapiens"))
    .collect()
)
hs_datasets = meta_hs['dataset_id'].unique().sort()
display(meta_hs)

data_hs = (
    pl.scan_parquet((RUN_DIR / "final" / "mavedb_all_curated_data_postfilter.parquet"))
    .filter(pl.col("dataset_id").is_in(hs_datasets))
    .collect()
)
display(data_hs)

meta_data_hs = (
    pl.scan_parquet((RUN_DIR / "final" / "mavedb_all_curated_data_with_metadata_postfilter.parquet"))
    .filter(pl.col("species").eq("Homo sapiens"))
    .collect()
)

display(meta_data_hs)

OUTPUT_METADATA_POSTVALIDATION_HS = (RUN_DIR / "final" / "mavedb_all_curated_metadata_postvalidation_homo_sapiens.parquet")
OUTPUT_DATA_POSTFILTER_HS = (RUN_DIR / "final" / "mavedb_all_curated_data_postfilter_homo_sapiens.parquet")
OUTPUT_METADATA_DATA_POSTVALIDATION_HS = (RUN_DIR / "final" / "mavedb_all_curated_metadata_data_postvalidation_homo_sapiens.parquet")

meta_hs.write_parquet(OUTPUT_METADATA_POSTVALIDATION_HS)
data_hs.write_parquet(OUTPUT_DATA_POSTFILTER_HS)
meta_data_hs.write_parquet(OUTPUT_METADATA_DATA_POSTVALIDATION_HS)

In [ ]:
# change to True when you're confident the data is ready to be published to BigQuery
PUBLISH_TO_BIGQUERY = False

if PUBLISH_TO_BIGQUERY:
    publish_to_bigquery(
        metadata_path=OUTPUT_METADATA_POSTVALIDATION_HS,
        data_path=OUTPUT_DATA_POSTFILTER_HS,
        project_id=BIGQUERY_PROJECT,
    )